# Gold: which patients surface, and exactly why

Computes the referral indicators, assigns every patient one of three states, and
measures whether the flag lands evenly across the cohort.

| | |
| --- | --- |
| **Reads** | `silver_*` |
| **Writes** | `gold_patient_signals`, `gold_criteria_hits`, `gold_criteria_definitions`, `gold_referral_state`, `gold_equity_check`, `gold_evidence` |

## Two tiers, not a score

A **sufficient** criterion surfaces a child on its own. A **contributory** one counts
only in combination - two or more. The criteria are not equally decisive: developmental
regression warrants a look by itself, one affected relative does not. Weighting them
equally surfaces a fifth of the clinic, and a list that long is a list nobody reads.

There is deliberately no risk score. A clinician reading *"surfaced because features
span four body systems"* can disagree with the threshold and say why. A clinician
reading *"risk 0.81"* can only defer or ignore.

## The thresholds below are placeholders

Every number in `CRITERIA` is a **placeholder pending sign-off by the SickKids genetics
service**. They are set to produce a demonstrable cohort, not to be clinically correct.
They are gathered in one cell, named, and written out as data alongside the results, so
that the conversation with the clinical team is about a table they can mark up rather
than about a number buried in code.

Nothing here is a referral decision. The output is a list of patients a clinician may
wish to look at, with the reasons attached.

## Three states that must never collapse

| State | Meaning | What it is **not** |
| --- | --- | --- |
| `indicators_present` | One or more criteria fired on the record | **Not a diagnosis**, and not a referral |
| `no_indicators_recorded` | Record read, no criterion fired | **Not "no indication"** - the record may be silent, not the child |
| `not_screened` | Too little record to read | **Not a clear screen** |

The middle state is the one that gets misread. A child with a genetic condition whose
features were simply never coded reads identically to a child without one. The pipeline
cannot tell them apart, so it does not claim to, and the agent is forbidden from
writing anything that suggests otherwise.

## Why equity is computed here and not left to a review

Case-finding built from historical patterns reproduces historical bias. If a service
has under-referred children whose families need an interpreter, criteria tuned on that
history will under-flag them again, and the flag will look objective while doing it.

So the flag rate is broken down by language and interpreter need **as an output**, and
the notebook asserts that neither attribute was available to the criteria.

Expect the assertion to pass and the rates to differ anyway. The cohort is generated so
that children whose families need an interpreter carry the same underlying rate of
clustered presentation, but fewer of their features reach the record. The disparity
arrives through the features themselves, not through the attribute.

That is the lesson: **a flag can be blind to a protected attribute and still reproduce
the inequity attached to it.** Excluding the column proves nothing; measuring the
outcome is the only thing that shows it. `validation_sensitivity.ipynb` then quantifies
what it costs in children missed.

Measuring it does not fix it. It makes it arguable - and it points at the real remedy,
which is interpreter-supported history-taking, not a better model.

In [ ]:
# ---- PLACEHOLDER THRESHOLDS -- pending sign-off by the genetics service ----
MIN_SYSTEMS_MULTI = 3
MIN_SPECIALTIES_ODYSSEY = 4
MIN_MONTHS_ODYSSEY = 12
MAX_DIAGNOSED_SHARE_ODYSSEY = 0.5
MIN_UNDIAGNOSED_ADMISSIONS = 2
MIN_CONTRIBUTORY = 2
PIPELINE_RUN_ID = ""

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (DoubleType, IntegerType, StringType, StructField,
                               StructType)

RUN_ID = PIPELINE_RUN_ID or "local"
WORKSPACE = spark.conf.get("trident.workspace.id")
SILVER = "silver_lakehouse"


def lake_table(lakehouse, table):
    return spark.read.format("delta").load(
        f"abfss://{WORKSPACE}@onelake.dfs.fabric.microsoft.com/"
        f"{lakehouse}.Lakehouse/Tables/{table}")


patients = lake_table(SILVER, "silver_patients")
encounters = lake_table(SILVER, "silver_encounters")
observations = lake_table(SILVER, "silver_observations")
family = lake_table(SILVER, "silver_family_history")
print("silver loaded")

In [ ]:
# ------------------------------------------------------- per-patient signals
# Phenotype.
phenotype = (observations.groupBy("patient_id").agg(
    F.countDistinct("hpo_id").alias("feature_count"),
    F.countDistinct("body_system").alias("systems_involved"),
    F.max(F.when(F.col("hpo_id") == "HP:0002376", 1).otherwise(0))
     .alias("has_regression"),
    F.max(F.when(F.col("body_system") == "neurodevelopment", 1).otherwise(0))
     .alias("has_neurodev"),
    F.min("observed_date").alias("first_feature_date")))

# Care pathway.
pathway = (encounters.groupBy("patient_id").agg(
    F.countDistinct("specialty").alias("distinct_specialties"),
    F.count("*").alias("encounter_count"),
    F.min("encounter_date").alias("first_encounter_date"),
    F.max("encounter_date").alias("last_encounter_date"),
    F.sum(F.when(F.col("diagnosis_recorded"), 1).otherwise(0))
     .alias("encounters_with_diagnosis"),
    F.sum(F.when(F.col("admitted") & ~F.col("diagnosis_recorded"), 1).otherwise(0))
     .alias("undiagnosed_admissions")))

pathway = (pathway
           .withColumn("months_in_service",
                       F.round(F.months_between(F.col("last_encounter_date"),
                                                F.col("first_encounter_date")), 1))
           .withColumn("diagnosed_share",
                       (F.col("encounters_with_diagnosis") /
                        F.col("encounter_count")).cast(DoubleType())))

signals = (patients
           .join(phenotype, "patient_id", "left")
           .join(pathway, "patient_id", "left")
           .join(family.drop("run_id"), "patient_id", "left"))

# A patient with no observations has zero features -- that is a genuine measured zero.
# A patient with no encounters has no pathway to measure, which is different, so those
# stay null and the criteria below treat null as "cannot assess".
signals = signals.fillna({"feature_count": 0, "systems_involved": 0,
                          "has_regression": 0, "has_neurodev": 0})

signals.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gold_patient_signals")
print(f"gold_patient_signals  {signals.count():,} rows")
signals.select("patient_id", "feature_count", "systems_involved",
               "distinct_specialties", "months_in_service", "diagnosed_share",
               "history_taken").show(6, truncate=False)

In [ ]:
# ------------------------------------------------------------- the criteria
# Each criterion is named, carries its own threshold, and is evaluated independently.
# No weighted score: a clinician reading "surfaced because features span 4 body systems"
# can disagree with the threshold. A clinician reading "risk score 0.81" cannot.

# Two tiers, because the criteria are not equally decisive. Developmental regression
# warrants a look on its own; one affected relative does not. Scoring them equally
# surfaces a fifth of the clinic, and a list that long is a list nobody reads.
CRITERIA = [
    ("MULTI_SYSTEM", "sufficient",
     f"Features recorded in {MIN_SYSTEMS_MULTI} or more body systems",
     F.col("systems_involved") >= MIN_SYSTEMS_MULTI),
    ("REGRESSION", "sufficient",
     "Developmental regression recorded (HP:0002376)",
     F.col("has_regression") == 1),
    ("NEURODEV_PLUS", "contributory",
     "A neurodevelopmental feature alongside a feature in another system",
     (F.col("has_neurodev") == 1) & (F.col("systems_involved") >= 2)),
    ("DIAGNOSTIC_ODYSSEY", "contributory",
     f"Seen by {MIN_SPECIALTIES_ODYSSEY}+ specialties over "
     f"{MIN_MONTHS_ODYSSEY}+ months with a diagnosis recorded at under "
     f"{MAX_DIAGNOSED_SHARE_ODYSSEY:.0%} of encounters",
     (F.col("distinct_specialties") >= MIN_SPECIALTIES_ODYSSEY) &
     (F.col("months_in_service") >= MIN_MONTHS_ODYSSEY) &
     (F.col("diagnosed_share") < MAX_DIAGNOSED_SHARE_ODYSSEY)),
    ("REPEAT_UNDIAGNOSED_ADMISSION", "contributory",
     f"{MIN_UNDIAGNOSED_ADMISSIONS}+ admissions with no diagnosis recorded",
     F.col("undiagnosed_admissions") >= MIN_UNDIAGNOSED_ADMISSIONS),
    ("FAMILY_HISTORY", "contributory",
     "Affected first-degree relative, consanguinity, or recurrent pregnancy loss",
     (F.col("history_taken") == True) &
     ((F.col("affected_first_degree") == True) |
      (F.col("consanguinity") == True) |
      (F.col("recurrent_pregnancy_loss") == True))),
]

# The criteria may only read these columns. Language and interpreter need are absent
# by construction, and this list is asserted against later.
PERMITTED_INPUTS = {
    "systems_involved", "has_neurodev", "has_regression", "distinct_specialties",
    "months_in_service", "diagnosed_share", "undiagnosed_admissions",
    "history_taken", "affected_first_degree", "consanguinity",
    "recurrent_pregnancy_loss",
}

hits = None
for name, tier, description, condition in CRITERIA:
    fired = (signals.filter(condition)
             .select("patient_id",
                     F.lit(name).alias("criterion"),
                     F.lit(tier).alias("tier"),
                     F.lit(description).alias("criterion_description")))
    hits = fired if hits is None else hits.unionByName(fired)

hits = hits.withColumn("run_id", F.lit(RUN_ID))
hits.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gold_criteria_hits")

print("criterion fire counts")
for row in (hits.groupBy("tier", "criterion").count()
            .orderBy("tier", F.desc("count")).collect()):
    print(f"  {row['tier']:13} {row['criterion']:30} {row['count']:>5,}")

# Write the thresholds out as data so the clinical conversation is about a table.
threshold_rows = [
    {"criterion": name, "tier": tier, "description": description, "run_id": RUN_ID,
     "status": "PLACEHOLDER - pending clinical sign-off"}
    for name, tier, description, _ in CRITERIA]
threshold_schema = StructType([
    StructField("criterion", StringType()),
    StructField("tier", StringType()),
    StructField("description", StringType()),
    StructField("run_id", StringType()),
    StructField("status", StringType())])
spark.createDataFrame(threshold_rows, threshold_schema).write.mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_criteria_definitions")
print("\nwrote gold_criteria_definitions")

In [ ]:
# ---------------------------------------------------------- the three states
fired = (spark.table("gold_criteria_hits").groupBy("patient_id").agg(
    F.count("*").alias("criteria_fired"),
    F.sum(F.when(F.col("tier") == "sufficient", 1).otherwise(0))
     .alias("sufficient_fired"),
    F.sum(F.when(F.col("tier") == "contributory", 1).otherwise(0))
     .alias("contributory_fired"),
    F.sort_array(F.collect_set("criterion")).alias("criteria")))

# One sufficient criterion surfaces the child. Contributory criteria only surface a
# child in combination -- a single one is a fact about the record, not a reason to act.
surfaced = ((F.col("sufficient_fired") > 0) |
            (F.col("contributory_fired") >= MIN_CONTRIBUTORY))

state = (signals.select("patient_id", "screenable", "record_days", "age_years",
                        "primary_language", "interpreter_required")
         .join(fired, "patient_id", "left")
         .fillna({"criteria_fired": 0, "sufficient_fired": 0,
                  "contributory_fired": 0})
         .withColumn("criteria", F.coalesce("criteria", F.array()))
         .withColumn("referral_state",
                     F.when(~F.col("screenable"), F.lit("not_screened"))
                      .when(surfaced, F.lit("indicators_present"))
                      .otherwise(F.lit("no_indicators_recorded")))
         .withColumn("run_id", F.lit(RUN_ID)))

state.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gold_referral_state")

total = state.count()
print(f"cohort {total:,}\n")
for row in state.groupBy("referral_state").count().orderBy(F.desc("count")).collect():
    print(f"  {row['referral_state']:24} {row['count']:>6,}  "
          f"({row['count'] / total:5.1%})")

print("\nof those with indicators, how many criteria fired:")
for row in (state.filter("referral_state = 'indicators_present'")
            .groupBy("criteria_fired").count().orderBy("criteria_fired").collect()):
    print(f"  {row['criteria_fired']} criteria   {row['count']:>5,}")

In [ ]:
# --------------------------------------------------------------- equity check
# First: prove the protected attributes were never inputs. This is an assertion about
# the code above, not a property of the data, so it is checked explicitly.
# Tokenise the parsed expression and intersect with the available columns, rather than
# pattern-matching each name into the string. Substring matching would miss
# `interpreter_required` sitting inside a longer token and report a clean pass on a
# genuine leak -- and a leak detector whose failure mode is a false pass is worse than
# none. Tokenising can only over-report, which fails safe.
import re

criteria_columns = set()
available = set(signals.columns)
for _, _, _, condition in CRITERIA:
    tokens = set(re.findall(r"[A-Za-z_][A-Za-z0-9_]*", str(condition)))
    criteria_columns |= tokens & available

leaked = criteria_columns - PERMITTED_INPUTS
if leaked:
    raise ValueError(
        f"criteria read columns outside the permitted set: {sorted(leaked)}. "
        f"Protected attributes must not reach the scoring inputs.")
print(f"criteria inputs verified: {sorted(criteria_columns)}")
print("primary_language / interpreter_required are NOT inputs\n")

# Second: measure whether the flag lands evenly anyway. Excluding an attribute from the
# inputs does not stop a proxy for it reaching them.
state = spark.table("gold_referral_state")
screened = state.filter("referral_state != 'not_screened'")

by_interpreter = (screened.groupBy("interpreter_required").agg(
    F.count("*").alias("screened"),
    F.sum(F.when(F.col("referral_state") == "indicators_present", 1).otherwise(0))
     .alias("flagged"))
    .withColumn("flag_rate", (F.col("flagged") / F.col("screened")).cast(DoubleType())))

print("flag rate by interpreter need")
for row in by_interpreter.orderBy("interpreter_required").collect():
    print(f"  interpreter={str(row['interpreter_required']):5}  "
          f"screened={row['screened']:>5,}  flagged={row['flagged']:>4,}  "
          f"rate={row['flag_rate']:.1%}")

by_language = (screened.groupBy("primary_language").agg(
    F.count("*").alias("screened"),
    F.sum(F.when(F.col("referral_state") == "indicators_present", 1).otherwise(0))
     .alias("flagged"))
    .withColumn("flag_rate", (F.col("flagged") / F.col("screened")).cast(DoubleType()))
    .filter(F.col("screened") >= 30))

print("\nflag rate by primary language (groups of 30+ screened)")
for row in by_language.orderBy(F.desc("flag_rate")).collect():
    print(f"  {row['primary_language']:14} screened={row['screened']:>5,}  "
          f"rate={row['flag_rate']:.1%}")

equity = (by_interpreter
          .withColumn("dimension", F.lit("interpreter_required"))
          .withColumn("group", F.col("interpreter_required").cast(StringType()))
          .select("dimension", "group", "screened", "flagged", "flag_rate")
          .unionByName(by_language
                       .withColumn("dimension", F.lit("primary_language"))
                       .withColumn("group", F.col("primary_language"))
                       .select("dimension", "group", "screened", "flagged",
                               "flag_rate"))
          .withColumn("run_id", F.lit(RUN_ID)))

equity.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gold_equity_check")
print("\nwrote gold_equity_check")

In [ ]:
# -------------------------------------------------- the evidence contract
# Every criterion that fired gets the specific records that support it, each with an ID
# the agent must cite. The agent computes nothing; it renders these and nothing else.

# Only patients the pipeline actually surfaced. gold_criteria_hits also holds the
# single contributory hits that did not meet the tier rule, and building evidence
# for those would hand the agent a patient it must not write about.
surfaced_ids = (spark.table("gold_referral_state")
                .filter("referral_state = 'indicators_present'")
                .select("patient_id"))

feature_evidence = (observations.alias("o")
    .join(surfaced_ids, "patient_id")
    .select("patient_id",
            F.concat(F.lit("OBS:"), F.col("observation_id")).alias("evidence_id"),
            F.lit("observation").alias("evidence_type"),
            F.col("observed_date").cast(StringType()).alias("evidence_date"),
            F.concat_ws(" ", F.col("hpo_label"),
                        F.concat(F.lit("("), F.col("hpo_id"), F.lit(")")),
                        F.lit("- system:"), F.col("body_system"))
             .alias("evidence_text")))

encounter_evidence = (encounters.alias("e")
    .join(surfaced_ids, "patient_id")
    .select("patient_id",
            F.concat(F.lit("ENC:"), F.col("encounter_id")).alias("evidence_id"),
            F.lit("encounter").alias("evidence_type"),
            F.col("encounter_date").cast(StringType()).alias("evidence_date"),
            F.concat_ws(" ", F.col("specialty"),
                        F.when(F.col("admitted"), F.lit("(admitted)"))
                         .otherwise(F.lit("(outpatient)")),
                        F.when(F.col("diagnosis_recorded"),
                               F.lit("- diagnosis recorded"))
                         .otherwise(F.lit("- no diagnosis recorded")))
             .alias("evidence_text")))

family_evidence = (family
    .filter(F.col("history_taken") == True)
    .join(surfaced_ids, "patient_id")
    .select("patient_id",
            F.concat(F.lit("FHX:"), F.col("patient_id")).alias("evidence_id"),
            F.lit("family_history").alias("evidence_type"),
            F.col("asked_on").cast(StringType()).alias("evidence_date"),
            F.concat_ws("; ",
                        F.when(F.col("affected_first_degree") == True,
                               F.lit("affected first-degree relative")),
                        F.when(F.col("consanguinity") == True,
                               F.lit("consanguinity recorded")),
                        F.when(F.col("recurrent_pregnancy_loss") == True,
                               F.lit("recurrent pregnancy loss")))
             .alias("evidence_text"))
    .filter(F.length(F.col("evidence_text")) > 0))

evidence = (feature_evidence.unionByName(encounter_evidence)
            .unionByName(family_evidence)
            .withColumn("run_id", F.lit(RUN_ID)))

evidence.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gold_evidence")

print(f"gold_evidence  {evidence.count():,} rows across "
      f"{evidence.select('patient_id').distinct().count():,} surfaced patients")
evidence.show(8, truncate=False)
print("\ngold complete")